# sensing-crit rescue — 临界传感 6-cell 同协议补跑（seed=7 道：s0_k4 + s1_k4 + s2_k4）

> 文档锚点：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) **§7.10**（取证缺口与缺失清单） · [`docs/rebrac_broad_validation_v2_seed43_supplement_plan.md`](../docs/rebrac_broad_validation_v2_seed43_supplement_plan.md) 附录 A · 论文章级验收 finding **E·M1**。
> 用户批准（2026-07-08）：六份云端 final_eval 本机不可达（Mac 不在身边），走**同协议补跑重取证**，与 §5.8 补种子并为一个执行轮。

本 notebook 是 3 道并行中的 **seed=7 道**（3 个 1M-step 训练单元，L4 约 7.5 h；`[skip]`/resume 支持跨 session 续跑）。
另两道：`sac_arrival_v2_sensing_crit_rescue_seed{0,7,42}.ipynb`。

## 九宫格现状（转录值 = 2026-06-18 云端确认；✅ = 本机 final_eval 已实核）

| 配置 | seed 0 | seed 7 | seed 42 |
|---|---|---|---|
| s0_k4 | 0.400 ✅ 已实核 | 0.267 ⚠ 待补跑 | 0.100 ✅ 已实核 |
| s1_k4 | 0.900 ⚠ 待补跑 | 0.800 ⚠ 待补跑 | 0.900 ✅ 已实核 |
| s2_k4 | 0.800 ⚠ 待补跑 | 0.733 ⚠ 待补跑 | 0.733 ⚠ 待补跑 |

## 预登记判读语义（跑前锁定）

- 补跑产出的 `final_eval.json` 即该 cell 的**新 ground truth**（可追溯证据）。
- 补跑值与转录值**逐位一致** → §7.10 该读数 ⚠→✅；六格全一致后 E·M1 勾销。
- **任何偏差（含 1 个 episode 之差）→ 先呈报**，`.tex` 刊值与 §7.10 一字不动，由用户裁决后统一处置。
- 本 notebook 只产出事实（per-cell 补跑值 + EXACT_MATCH 布尔），不做任何论文侧改动。

## 协议（与 §7.1 / §7.6 / §7.7 逐项一致，唯一变量 = probe layout × seed）

`vanilla SAC / arrival_v2 / k=4 / 1M steps / num_envs=6 / random_steps=update_after=5000 /`
`batch=256 / hidden=256 / eval_every=25k×30ep / 终检 final_eval 30 deterministic ep /`
`flow=wake_v8_U1p50_Re250 / manifest=benchmarks/single_u15_cross_tgt15.json（固定，禁止再生成）`

## 0. GPU sanity

In [ ]:
!nvidia-smi | head -10

## 1. Mount Drive + cwd

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR

## 2. Config — 本道 RUNS 矩阵（路径一律相对仓库根，回避 Drive 路径含空格的 `!python {var}` 拆断 bug）

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== 与 §7.1/§7.6/§7.7 严格一致（除 probe layout / seed 外零新变量）====
OBJECTIVE = 'arrival_v2'
HISTORY_LENGTH = 4
TARGET_SPEED = 1.5
SEED = 7

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
TOTAL_STEPS = 1_000_000
DEVICE = 'cuda'

# 显式拒绝所有 SAC 改进项 — vanilla（与被补的 6 cell 原协议一致）
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
BENCHMARK_KEY = 'single_u15_cross_tgt15'
TASK_GEOMETRY = 'cross_stream'
MANIFEST_PATH = Path(f'benchmarks/{BENCHMARK_KEY}.json')

RUNS = [
    {'probe': 's0', 'expected': 0.267, 'obs_dim': 48, 'run_root': 'experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_7', 'ckpt_root': 'checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_7'},
    {'probe': 's1', 'expected': 0.8, 'obs_dim': 56, 'run_root': 'experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_7', 'ckpt_root': 'checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_7'},
    {'probe': 's2', 'expected': 0.733, 'obs_dim': 72, 'run_root': 'experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s2_k4/seed_7', 'ckpt_root': 'checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s2_k4/seed_7'},
]

SUMMARY_DIR = Path('experiments/arrival_v2_prototype/sensing_crit_rescue_summary')
VERDICT_JSON = SUMMARY_DIR / 'rescue_verdict_seed7.json'

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'SEED = {SEED}   runs = ' + ', '.join(r['probe'] + '_k4' for r in RUNS))
for r in RUNS:
    print(f"  {r['probe']}_k4  expected(transcribed)={r['expected']:.3f}  "
          f"expected obs_dim={r['obs_dim']}  -> {r['run_root']}")

## 3. Preflight — flow / 固定 manifest / 已实核锚点 / 目标位状态

In [ ]:
fp = Path(SINGLE_FLOW)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')

# manifest 是固定评估集（git 内）；缺失说明 Drive 同步不完整——禁止在此再生成
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'fixed manifest missing: {MANIFEST_PATH} — sync repo to Drive first; DO NOT regenerate')
print(f'[OK] fixed manifest: {MANIFEST_PATH}')

In [ ]:
# 已实核 ✅ 锚点（§7.10）——就位即打印，用于 sanity（不参与本道训练）
ANCHORS = [
    ('s0_k4/seed_0',  'experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_0',  0.400),
    ('s0_k4/seed_42', 'experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42', 0.100),
    ('s1_k4/seed_42', 'experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42', 0.900),
]
for label, root, ref in ANCHORS:
    f = Path(root) / 'results' / 'final_eval.json'
    if f.exists():
        v = float(json.loads(f.read_text(encoding='utf-8'))['eval_success_rate'])
        mark = '✓' if abs(v - ref) < 1e-9 else f'✗ MISMATCH vs {ref}'
        print(f'[OK] anchor {label}: final={v:.3f} {mark}')
    else:
        print(f'[WARN] anchor {label} missing on Drive: {f}')

# 目标位：六缺格的本道子集应为空（若 Mac 文件已先行同步，[skip] 会自动跳过）
for r in RUNS:
    f = Path(r['run_root']) / 'results' / 'final_eval.json'
    print(('[PRESENT — 将被 [skip] 保护，不覆盖] ' if f.exists() else '[EMPTY — 待补跑] ') + str(f))

## 4. Train — s0_k4 + s1_k4 + s2_k4 × seed=7（各 1M steps，`[skip]`/resume 跨 session 安全）

`[skip]` 判定：`trainer_state.json` 的 `env_step >= 1M`；中断后重跑本 cell 自动 `--resume`。

In [ ]:
for r in RUNS:
    run_root = Path(r['run_root'])
    run_root_s = str(run_root)
    ckpt_root_s = r['ckpt_root']
    probe = r['probe']
    manifest_s = str(MANIFEST_PATH)

    state_path = run_root / 'trainer_state.json'
    current_step = 0
    if state_path.exists():
        current_step = int(json.loads(state_path.read_text(encoding='utf-8')).get('env_step', 0))
    print(f'\n========== {probe}_k4 seed={SEED}: env_step={current_step:,} / {TOTAL_STEPS:,} ==========')

    if current_step >= TOTAL_STEPS:
        print(f'[skip] already trained to {current_step:,}')
        continue
    elif current_step > 0:
        print(f'[resume] continuing from {current_step:,}')
        !python -u -m scripts.train_sac \
            --resume {run_root_s} \
            --total-steps {TOTAL_STEPS} \
            --eval-every {EVAL_EVERY} \
            --eval-episodes {EVAL_EPISODES} \
            --checkpoint-every {CHECKPOINT_EVERY} \
            --eval-manifest {manifest_s} \
            --device {DEVICE}
    else:
        print('[train] fresh start')
        !python -u -m scripts.train_sac \
            --flow {SINGLE_FLOW} \
            --task-geometry {TASK_GEOMETRY} \
            --target-speed {TARGET_SPEED} \
            --objective {OBJECTIVE} \
            --probe-layout {probe} \
            --history-length {HISTORY_LENGTH} \
            --total-steps {TOTAL_STEPS} \
            --random-steps {RANDOM_STEPS} \
            --update-after {UPDATE_AFTER} \
            --batch-size {BATCH_SIZE} \
            --hidden-dim {HIDDEN_DIM} \
            --num-envs {NUM_ENVS} \
            --eval-every {EVAL_EVERY} \
            --eval-episodes {EVAL_EPISODES} \
            --checkpoint-every {CHECKPOINT_EVERY} \
            --eval-manifest {manifest_s} \
            --seed {SEED} \
            --device {DEVICE} \
            --save-dir {run_root_s} \
            --checkpoint-dir {ckpt_root_s}

## 5. Verdict — 补跑值 vs 转录值（纯事实输出，不判读、不改论文侧）

In [ ]:
records = []
for r in RUNS:
    run_root = Path(r['run_root'])
    fe_path = run_root / 'results' / 'final_eval.json'
    tc_path = run_root / 'results' / 'train_config.txt'
    if not fe_path.exists():
        print(f"[WARN] {r['probe']}_k4: final_eval.json 缺失（训练未完成？）")
        continue
    d = json.loads(fe_path.read_text(encoding='utf-8'))
    succ = float(d['eval_success_rate'])
    n_ep = int(float(d.get('num_eval_episodes', 0)))
    counts = d.get('eval_termination_counts', {})

    obs_dim_cfg = None
    if tc_path.exists():
        for ln in tc_path.read_text(encoding='utf-8').splitlines():
            if ln.strip().startswith('obs_dim='):
                obs_dim_cfg = int(ln.strip().split('=', 1)[1])
    obs_ok = (obs_dim_cfg == r['obs_dim'])

    # 转录值是三位小数舍入（如 8/30→0.267），final_eval 是全精度；
    # 「逐位一致」按成功回合数比较，不做浮点直等。
    n_succ = round(succ * n_ep) if n_ep else -1
    expected_n_succ = round(r['expected'] * 30)
    exact = (n_ep == 30) and (n_succ == expected_n_succ)
    records.append({
        'probe': r['probe'], 'seed': SEED,
        'rescue_success': succ, 'rescue_n_succ': n_succ,
        'transcribed': r['expected'], 'transcribed_n_succ': expected_n_succ,
        'exact_match': exact, 'num_eval_episodes': n_ep,
        'termination_counts': counts,
        'obs_dim': obs_dim_cfg, 'obs_dim_expected': r['obs_dim'], 'obs_dim_ok': obs_ok,
        'final_eval_path': str(fe_path),
    })
    mark = ('EXACT_MATCH ✓' if exact
            else f'MISMATCH ✗ ({n_succ}/{n_ep} vs {expected_n_succ}/30) → 呈报')
    print(f"{r['probe']}_k4 seed={SEED}: rescue={succ:.3f} ({n_succ}/{n_ep})  "
          f"transcribed={r['expected']:.3f} ({expected_n_succ}/30)"
          f"  [{mark}]  counts={counts}  obs_dim={obs_dim_cfg} ({'ok' if obs_ok else 'UNEXPECTED'})")

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
VERDICT_JSON.write_text(json.dumps({
    'notebook': 'sac_arrival_v2_sensing_crit_rescue_seed7',
    'protocol': 'vanilla SAC / arrival_v2 / k4 / 1M / num_envs=6 / final_eval 30ep / '
                'flow wake_v8_U1p50_Re250 / manifest single_u15_cross_tgt15',
    'semantics': 'rescue final_eval is the new ground truth; any deviation from the '
                 'transcribed 2026-06-18 value must be escalated before touching thesis numbers',
    'runs': records,
}, indent=1, ensure_ascii=False), encoding='utf-8')
print(f'\n[saved] {VERDICT_JSON}')
print('all EXACT_MATCH:', bool(records) and all(x['exact_match'] for x in records),
      f'({len(records)}/{len(RUNS)} runs evaluated)')

## 6. 跑完后清单（回到 local）

三道全部收齐后（`rescue_verdict_seed0/7/42.json` × 3）：

1. 只取回轻量结果（**不要**取回 `state/replay_latest.pkl`，每个 ~GB 级）：

```bash
# 在本地 repo root（对本道涉及的每个 run dir）：
rsync -av --exclude 'state/' --exclude 'logs/' \
  '<drive>/experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/<layout>_k4/seed_7/' \
  experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/<layout>_k4/seed_7/
rsync -av '<drive>/experiments/arrival_v2_prototype/sensing_crit_rescue_summary/' experiments/arrival_v2_prototype/sensing_crit_rescue_summary/
```

2. 回读判读入口：`paper/thesis_ch5/next_session_prompt.md`（执行轮回读节）。
   六格全 EXACT_MATCH → §7.10 ⚠→✅ + E·M1 勾销；任一 MISMATCH → 呈报硬停。